# 🚀 LOGI-LLM: Adaptador Gemma-4-E4B para Google Colab com Interface Gradio e Obsidian API

Este notebook foi desenvolvido para rodar o modelo **Gemma-4-E4B-Uncensored** com aceleração por GPU (T4/A100) gratuita no Google Colab.

Ele contém duas opções de uso:
1. **Interface de Chat Web (Gradio)**: Chat interativo de alta performance no navegador ou celular.
2. **Servidor API do Obsidian**: Transforma a LLM em uma API compatível com a OpenAI, permitindo que você digite e receba respostas em tempo real diretamente dentro das notas do seu **Obsidian**!

### ⚠️ **PASSO IMPORTANTE ANTES DE COMEÇAR**:
Certifique-se de que a sua máquina do Colab está com suporte a GPU ativo:
1. No menu superior, clique em **Ambiente de execução** (Runtime) > **Alterar tipo de ambiente de execução** (Change runtime type).
2. Em **Acelerador de hardware** (Hardware accelerator), selecione **GPU T4** (ou superior).
3. Clique em **Salvar**.

## 🛠️ Passo 1: Instalação das Dependências

Para extrair a máxima velocidade da placa de vídeo gratuita do Google Colab, vamos instalar o `llama-cpp-python` com suporte nativo à placa de vídeo (CUDA).

Oferecemos duas opções abaixo:
- **Opção 1 (Recomendada)**: Instala instantaneamente (15 segundos) usando pacotes pré-compilados.
- **Opção 2**: Compila direto do código-fonte caso a primeira dê algum erro de compatibilidade.

In [ ]:
# OPÇÃO 1: RÁPIDA (Instala em 15 segundos usando a versão pré-compilada para CUDA 12.1/12.2 com suporte a servidor)
!pip install "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# OPÇÃO 2: UNIVERSAL (Demora cerca de 3 a 5 minutos pois compila tudo do zero)
# !CMAKE_ARGS="-DLLAMA_CUDA=on" FORCE_CMAKE=1 pip install "llama-cpp-python[server]" --no-cache-dir

# Instalar Gradio para a interface web e Hugging Face Hub
!pip install gradio huggingface-hub

## ⚡ Passo 2: Modo 1 - Inicializar a Interface Web de Chat (Gradio)

Use esta célula se quiser conversar com o modelo diretamente no navegador ou no celular via link público do Gradio.

In [ ]:
from llama_cpp import Llama
import gradio as gr

# 1. Inicializa o modelo com suporte a GPU
print("📥 Baixando e carregando o modelo na GPU...")
llm = Llama.from_pretrained(
    repo_id="HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive",
    filename="Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-IQ3_M.gguf",
    n_ctx=2048,       # Janela de contexto de 2048 tokens
    n_gpu_layers=-1,  # Descarrega 100% das camadas na GPU T4
    use_mmap=False    # Desativa mmap para evitar picos de memória RAM no Colab
)
print("✅ Modelo pronto na placa de vídeo!")

# 2. Função de chat que suporta streaming em tempo real
def chat_respond(message, history, system_prompt, temperature, max_tokens, top_p, repeat_penalty):
    messages = []
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt})
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})
    
    try:
        response_stream = llm.create_chat_completion(
            messages=messages,
            temperature=float(temperature),
            max_tokens=int(max_tokens),
            top_p=float(top_p),
            repeat_penalty=float(repeat_penalty),
            stream=True
        )
        partial_text = ""
        for chunk in response_stream:
            delta = chunk['choices'][0]['delta']
            if 'content' in delta:
                partial_text += delta['content']
                yield partial_text
    except Exception as e:
        yield f"⚠️ Ocorreu um erro ao gerar a resposta: {str(e)}"

# 3. Criando a estrutura visual com o tema Soft
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="indigo")) as demo:
    with gr.Row():
        gr.Markdown(
            """
            # 🚀 LOGI-LLM: Gemma-4-E4B Uncensored
            ### Interface Inteligente rodando com Aceleração de GPU T4 no Google Colab!
            """
        )
    with gr.Row():
        with gr.Column(scale=3, variant="panel"):
            gr.Markdown("### ⚙️ Parâmetros da IA")
            system_prompt = gr.Textbox(
                value="Você é um assistente de IA prestativo, inteligente e responde de forma clara em Português.",
                label="System Prompt (Personalidade)",
                lines=4
            )
            with gr.Accordion("⚙️ Ajustes Avançados", open=True):
                temperature = gr.Slider(minimum=0.1, maximum=1.5, value=0.7, step=0.05, label="Temperatura (Criatividade)")
                max_tokens = gr.Slider(minimum=64, maximum=2048, value=512, step=64, label="Max Tokens (Tamanho da resposta)")
                top_p = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top P")
                repeat_penalty = gr.Slider(minimum=1.0, maximum=1.5, value=1.1, step=0.05, label="Penalidade de Repetição")
            gr.HTML("<hr style='border: 0; border-top: 1px solid rgba(255,255,255,0.1); margin: 15px 0;'>")
            gr.Markdown("⚡ **Aceleração**: CUDA GPU active\n💾 **Modelo**: Gemma-4-E4B")
        with gr.Column(scale=7):
            chatbot = gr.ChatInterface(
                fn=chat_respond,
                additional_inputs=[system_prompt, temperature, max_tokens, top_p, repeat_penalty],
                chatbot=gr.Chatbot(height=520, placeholder="🤖 **Como posso ajudar você hoje? Digite qualquer pergunta!**"),
                textbox=gr.Textbox(placeholder="Digite sua mensagem aqui...", container=False, scale=7),
                submit_btn="Enviar 🚀",
                retry_btn="Tentar Novamente 🔄",
                clear_btn="Limpar Chat 🗑️",
            )

demo.queue().launch(share=True, debug=True)

## ⚡ Passo 3: Modo 2 - Servidor API para Integração com o Obsidian 📂

Rode esta célula caso você queira usar o modelo diretamente **dentro do seu Obsidian** (através de plugins como **Copilot** ou **BMO Chatbot**).

Esta célula irá:
1. Iniciar o servidor de API compatível com a OpenAI na porta `8000` em segundo plano.
2. Expor a porta `8000` publicamente via LocalTunnel, fornecendo uma URL segura (`https://xxxx.loca.lt`).

### ⚙️ Como configurar no Obsidian:
1. Instale o plugin **Copilot** ou **BMO Chatbot** no seu Obsidian.
2. Vá nas configurações do plugin, mude o **Provider** (Provedor) para **Custom / OpenAI Compatible**.
3. No campo **API URL / Endpoint**, cole a URL gerada pelo LocalTunnel abaixo (ex: `https://slimy-wolves-giggle.loca.lt/v1`). **Importante**: Certifique-se de incluir `/v1` no final da URL!
4. No campo **API Key**, digite qualquer coisa (ex: `123`).
5. No campo **Model Name**, selecione ou escreva `Gemma` ou `default`.
6. Abra a barra de chat lateral no Obsidian e divirta-se conversando com a LLM do Colab direto nas suas notas!

In [ ]:
# 1. Instalar localtunnel se não estiver instalado para criar o túnel público
!npm install -g localtunnel

# 2. Baixar o arquivo do modelo e obter o caminho local do cache
import subprocess
import time
from huggingface_hub import hf_hub_download

print("📥 Baixando/Verificando modelo Gemma-4-E4B...")
model_path = hf_hub_download(
    repo_id="HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive",
    filename="Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-IQ3_M.gguf"
)

print("⚡ Iniciando Servidor API de Alta Performance (OpenAI-Compatible) na porta 8000...")
# Inicia o servidor em segundo plano descarregando tudo na GPU (-1 layers)
server_process = subprocess.Popen([
    "python", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "-1",
    "--n_ctx", "2048",
    "--port", "8000"
])

# Aguarda o servidor levantar
time.sleep(6)
print("✅ Servidor API Ativo!")

print("🌐 Expondo o servidor para o Obsidian via LocalTunnel...")
print("⚠️ IMPORTANTE: Se o LocalTunnel pedir um 'IP password', cole o IP público do seu Colab!")
print("(Você pode descobrir o IP do Colab rodando '!curl ipv4.icanhazip.com' em uma célula temporária)")
print("⚠️ LEMBRE-SE de colar '/v1' no final da URL no Obsidian!\n")

# Cria o túnel na porta 8000 exposto publicamente
!npx localtunnel --port 8000